In [1]:
# HealthGuard AI - Diabetes Data Cleaning
# Author: HealthGuard AI Team
# Date: 2026
# Techniques: IQR Outlier Removal, Median Imputation,
#             Standard Scaling, SMOTE

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
%matplotlib inline

plt.style.use('seaborn-v0_8')

print("=" * 50)
print("  HealthGuard AI - Diabetes Data Cleaning")
print("=" * 50)
print("Libraries Loaded Successfully!")

  HealthGuard AI - Diabetes Data Cleaning
Libraries Loaded Successfully!


In [2]:
# Load Raw Diabetes Dataset

df = pd.read_csv("E:/HealthGuard_AI/data/raw/diabetes.csv")

print(f"Original Shape: {df.shape}")
print(f"Total Missing Values: {df.isnull().sum().sum()}")
df.head()

Original Shape: (768, 9)
Total Missing Values: 0


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
# Step 1: Fix Zero Values
# These are actually missing values in medical data

print("=" * 50)
print("STEP 1: FIX ZERO VALUES")
print("=" * 50)

# Columns where 0 means missing
zero_cols = ['Glucose', 'BloodPressure',
             'SkinThickness', 'Insulin', 'BMI']

print("Before fixing zeros:")
for col in zero_cols:
    zeros = (df[col] == 0).sum()
    print(f"{col}: {zeros} zeros ({zeros/len(df)*100:.1f}%)")

# Replace 0 with NaN
df[zero_cols] = df[zero_cols].replace(0, np.nan)

print("\nAfter replacing with NaN:")
print(df[zero_cols].isnull().sum())

STEP 1: FIX ZERO VALUES
Before fixing zeros:
Glucose: 5 zeros (0.7%)
BloodPressure: 35 zeros (4.6%)
SkinThickness: 227 zeros (29.6%)
Insulin: 374 zeros (48.7%)
BMI: 11 zeros (1.4%)

After replacing with NaN:
Glucose            5
BloodPressure     35
SkinThickness    227
Insulin          374
BMI               11
dtype: int64


In [4]:
# Step 2: Handle Missing Values
# Using Median Imputation - Best for medical data
# Reason: Median is not affected by outliers

print("=" * 50)
print("STEP 2: HANDLE MISSING VALUES")
print("=" * 50)

# Group by Outcome for better imputation
for col in zero_cols:
    df[col] = df.groupby('Outcome')[col].transform(
        lambda x: x.fillna(x.median()))

print("Missing Values After Imputation:")
print(df.isnull().sum())
print("\nMedian Imputation Complete!")

STEP 2: HANDLE MISSING VALUES
Missing Values After Imputation:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

Median Imputation Complete!


In [5]:
# Step 3: Remove Duplicate Rows

print("=" * 50)
print("STEP 3: REMOVE DUPLICATES")
print("=" * 50)

before = len(df)
df = df.drop_duplicates()
after = len(df)

print(f"Before: {before} rows")
print(f"After: {after} rows")
print(f"Removed: {before - after} duplicates")

STEP 3: REMOVE DUPLICATES
Before: 768 rows
After: 768 rows
Removed: 0 duplicates


In [6]:
# Step 4: Remove Outliers Using IQR Method
# IQR = Best technique for medical data
# Remove values below Q1-1.5*IQR and above Q3+1.5*IQR

print("=" * 50)
print("STEP 4: REMOVE OUTLIERS (IQR METHOD)")
print("=" * 50)

before = len(df)

# Apply IQR only on these columns
outlier_cols = ['Glucose', 'BloodPressure', 'SkinThickness',
                'Insulin', 'BMI', 'DiabetesPedigreeFunction']

for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: {outliers} outliers removed")

    df = df[(df[col] >= lower) & (df[col] <= upper)]

after = len(df)
print(f"\nBefore: {before} rows")
print(f"After: {after} rows")
print(f"Total Removed: {before - after} outliers")

STEP 4: REMOVE OUTLIERS (IQR METHOD)
Glucose: 0 outliers removed
BloodPressure: 14 outliers removed
SkinThickness: 85 outliers removed
Insulin: 41 outliers removed
BMI: 6 outliers removed
DiabetesPedigreeFunction: 27 outliers removed

Before: 768 rows
After: 595 rows
Total Removed: 173 outliers


In [7]:
# Step 5: Feature Scaling Using StandardScaler
# Reason: Makes all features same scale
# ML models work better with scaled data

print("=" * 50)
print("STEP 5: FEATURE SCALING")
print("=" * 50)

# Separate features and target
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Apply Standard Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("Before Scaling:")
print(df[['Glucose', 'BMI', 'Age']].describe().round(2))

print("\nAfter Scaling:")
print(X_scaled[['Glucose', 'BMI', 'Age']].describe().round(2))
print("\nScaling Complete!")

STEP 5: FEATURE SCALING
Before Scaling:
       Glucose     BMI     Age
count   595.00  595.00  595.00
mean    118.48   31.75   33.46
std      28.78    6.01   11.87
min      44.00   18.20   21.00
25%      99.00   27.40   24.00
50%     114.00   31.60   29.00
75%     136.00   35.50   41.00
max     196.00   47.90   81.00

After Scaling:
       Glucose     BMI     Age
count   595.00  595.00  595.00
mean      0.00   -0.00   -0.00
std       1.00    1.00    1.00
min      -2.59   -2.26   -1.05
25%      -0.68   -0.72   -0.80
50%      -0.16   -0.03   -0.38
75%       0.61    0.62    0.64
max       2.70    2.69    4.01

Scaling Complete!


In [8]:
# Step 6+7: Train Test Split FIRST, then SMOTE only on Training data
# This prevents data leakage - test set stays truly unseen

print("=" * 50)
print("STEP 6: TRAIN TEST SPLIT (BEFORE SMOTE)")
print("=" * 50)

X_train_raw, X_test, y_train_raw, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y)

print(f"Train (before SMOTE): {len(X_train_raw)}")
print(f"Test (untouched): {len(X_test)}")

print("\n" + "=" * 50)
print("STEP 7: SMOTE ON TRAINING DATA ONLY")
print("=" * 50)

print("Before SMOTE:")
print(f"No Diabetes (0): {(y_train_raw==0).sum()}")
print(f"Diabetes (1): {(y_train_raw==1).sum()}")

smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train_raw, y_train_raw)

print("\nAfter SMOTE:")
print(f"No Diabetes (0): {(y_train==0).sum()}")
print(f"Diabetes (1): {(y_train==1).sum()}")
print(f"\nFinal Training Samples: {len(X_train)}")
print(f"Final Testing Samples: {len(X_test)} (never touched by SMOTE)")

STEP 6: TRAIN TEST SPLIT (BEFORE SMOTE)
Train (before SMOTE): 476
Test (untouched): 119

STEP 7: SMOTE ON TRAINING DATA ONLY
Before SMOTE:
No Diabetes (0): 322
Diabetes (1): 154

After SMOTE:
No Diabetes (0): 322
Diabetes (1): 322

Final Training Samples: 644
Final Testing Samples: 119 (never touched by SMOTE)


In [9]:
# Step 8: Save All Cleaned Data

print("=" * 50)
print("STEP 8: SAVE CLEANED DATA")
print("=" * 50)

# Save cleaned full dataset
df_cleaned = pd.concat([X_scaled,
                        y.reset_index(drop=True)], axis=1)
df_cleaned.to_csv(
    "E:/HealthGuard_AI/data/processed/diabetes_cleaned_final.csv",
    index=False)

# Save train test splits
X_train.to_csv(
    "E:/HealthGuard_AI/data/processed/diabetes_X_train.csv",
    index=False)
X_test.to_csv(
    "E:/HealthGuard_AI/data/processed/diabetes_X_test.csv",
    index=False)
y_train.to_csv(
    "E:/HealthGuard_AI/data/processed/diabetes_y_train.csv",
    index=False)
y_test.to_csv(
    "E:/HealthGuard_AI/data/processed/diabetes_y_test.csv",
    index=False)

print("Files Saved:")
print(" diabetes_cleaned_final.csv")
print(" diabetes_X_train.csv")
print(" diabetes_X_test.csv")
print(" diabetes_y_train.csv")
print(" diabetes_y_test.csv")
print(f"\nLocation: E:/HealthGuard_AI/data/processed/")

STEP 8: SAVE CLEANED DATA
Files Saved:
 diabetes_cleaned_final.csv
 diabetes_X_train.csv
 diabetes_X_test.csv
 diabetes_y_train.csv
 diabetes_y_test.csv

Location: E:/HealthGuard_AI/data/processed/


In [11]:
# Cleaning Summary Report

print("=" * 60)
print("   DIABETES CLEANING - SUMMARY REPORT")
print("=" * 60)

print("\nTECHNIQUES USED:")
print("-" * 40)
print("1. Zero Value Treatment - Replaced with NaN")
print("2. Median Imputation - Grouped by Outcome")
print("3. Duplicate Removal")
print("4. IQR Outlier Removal")
print("5. Standard Scaling")
print("6. Train Test Split BEFORE SMOTE (leakage-free)")
print("7. SMOTE - Applied only on Training Data")

print("\nRESULTS:")
print("-" * 40)
print(f"Original Rows: 768")
print(f"After Cleaning: {len(df)}")
print(f"Training Set (after SMOTE): {len(X_train)}")
print(f"Testing Set (untouched, real): {len(X_test)}")

print("\nDiabetes Cleaning Complete!")
print("=" * 60)

   DIABETES CLEANING - SUMMARY REPORT

TECHNIQUES USED:
----------------------------------------
1. Zero Value Treatment - Replaced with NaN
2. Median Imputation - Grouped by Outcome
3. Duplicate Removal
4. IQR Outlier Removal
5. Standard Scaling
6. Train Test Split BEFORE SMOTE (leakage-free)
7. SMOTE - Applied only on Training Data

RESULTS:
----------------------------------------
Original Rows: 768
After Cleaning: 595
Training Set (after SMOTE): 644
Testing Set (untouched, real): 119

Diabetes Cleaning Complete!


In [12]:
# Save Scaler for Web App
import pickle

scaler_path = "E:/HealthGuard_AI/models/saved/diabetes_scaler.pkl"
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

print(" Diabetes Scaler Saved!")
print(f"Location: {scaler_path}")

 Diabetes Scaler Saved!
Location: E:/HealthGuard_AI/models/saved/diabetes_scaler.pkl
